In [1]:
import polars as pl
import numpy as np
import pandas as pd

from ohlc_dss_model.data import (
    load_parquet, remove_incomplete_days, write_parquet,
    intraday_session_tagging, session_tagging, 
    filter_valid_sessions
)

from ohlc_dss_model.features import(
    aggregate_sessions, yang_zhang,
    PRE_NY_SPEC, FULL_DAY_SPEC,
    calculate_excursion_bands, assign_direction,
    detect_pivots, pivot_extraction, build_pivot_features,
    build_event_table, encode_news_context, inspect_event_table
)

from ohlc_dss_model.utils import (
    convert_to_timezone, plot_session
)

import os
from dotenv import load_dotenv

from ohlc_dss_model.config import config

from datetime import date

In [2]:
raw_data = load_parquet()
raw_data = convert_to_timezone(raw_data)
raw_data = session_tagging(raw_data)
raw_data = intraday_session_tagging(raw_data)
raw_data = remove_incomplete_days(raw_data)

aggregated_data = aggregate_sessions(raw_data)
aggregated_data = filter_valid_sessions(aggregated_data)

aggregated_data = aggregated_data.with_columns(
    pl.col("O_Asia").alias("O_Ref")
)

aggregated_data = yang_zhang(aggregated_data, FULL_DAY_SPEC, mode="historical")
aggregated_data = yang_zhang(aggregated_data, PRE_NY_SPEC, mode="today")

aggregated_data = assign_direction(aggregated_data)
aggregated_data = calculate_excursion_bands(aggregated_data)
# aggregated_data = aggregated_data.drop_nulls()

pivots_data = detect_pivots(raw_data, n=1)
pivots_data = pivot_extraction(pivots_data)
pivots_data = build_pivot_features(pivots_data, aggregated_data)

In [3]:
print(pivots_data.columns)
pivots_data.filter(pl.col("Session") == date(2026, 2, 4)).select([
    "DateTime", "delta_Pi_k", "delta_b_k", "Speed_k", "Dir_k", "Turn_k", "P_k", "Pi_k"
]).head(8)

['DateTime', 'Intraday_Session', 'Session', 'P_k', 's_k', 'O_New York', 'O_Asia', 'O_London', 'H_New York', 'H_Asia', 'H_London', 'L_New York', 'L_Asia', 'L_London', 'C_New York', 'C_Asia', 'C_London', 'O_Ref', 'Sigma_Historical', 'Sigma_Today', 'Z_Body', 'Z_Sigma', 'Tau', 'Direction', '_delta_t', 'Band_AE_Pos_Center', 'Band_AE_Neg_Center', 'Band_FE_Pos_Center', 'Band_FE_Neg_Center', 'Band_AE_Neg_Upper', 'Band_AE_Neg_Lower', 'Band_AE_Pos_Upper', 'Band_AE_Pos_Lower', 'Band_FE_Neg_Upper', 'Band_FE_Neg_Lower', 'Band_FE_Pos_Upper', 'Band_FE_Pos_Lower', 'Pi_k', 'Sigma_Price', 'Delta_FE_Pos', 'Delta_AE_Pos', 'Delta_FE_Neg', 'Delta_AE_Neg', 'State_AE_Neg', 'State_AE_Pos', 'State_FE_Neg', 'State_FE_Pos', 'delta_Pi_k', 'delta_b_k', 'Speed_k', 'Dir_k', 'Turn_k']


DateTime,delta_Pi_k,delta_b_k,Speed_k,Dir_k,Turn_k,P_k,Pi_k
"datetime[ns, America/New_York]",f64,i16,f64,i32,i8,f64,f64
2026-02-03 19:00:00 EST,0.0,0,0.0,0,0,25435.75,0.160788
2026-02-03 19:30:00 EST,-0.199944,1,-0.199944,-1,0,25375.75,-0.039156
2026-02-03 21:00:00 EST,0.326576,3,0.108859,1,1,25473.75,0.28742
2026-02-03 21:30:00 EST,-0.247431,1,-0.247431,-1,1,25399.5,0.039989
2026-02-03 22:30:00 EST,0.174118,2,0.087059,1,1,25451.75,0.214107
2026-02-03 23:30:00 EST,-0.012497,2,-0.006248,-1,1,25448.0,0.20161
2026-02-04 00:30:00 EST,-0.130797,2,-0.065398,-1,0,25408.75,0.070814
2026-02-04 02:30:00 EST,0.351569,4,0.087892,1,1,25514.25,0.422382


***
### **Dataset Contract**
This section will be utilized the transform the already engineered feature datas into a dataset for transformer.

Before moving on we will be defining which features are to be included in the dataset this is done for us not to have any headache later

In [4]:
# Defining pivot column whitelists

# configurable in config.py
MAX_PIVOT = 27

PIVOT_NUMERIC_WHITELIST = [
    "Pi_k",
    'Delta_FE_Pos', 'Delta_AE_Pos', 'Delta_FE_Neg', 'Delta_AE_Neg',
    'State_AE_Neg', 'State_AE_Pos', 'State_FE_Neg', 'State_FE_Pos',
    "delta_Pi_k", "delta_b_k", "Speed_k", "Dir_k", "Turn_k",
]

PIVOT_CATEGORICAL_WHITELIST = ["s_k", "Intraday_Session"]

In [5]:
_df = aggregated_data.with_columns(
    pl.col("Sigma_Historical").shift(1).alias("Sigma_Historical_Shifted")
)

#### **Normalizing Session OHLC Data**
Since we are feeding it into the context hence we will need to normalize it so it is consistent across all regime

$$\sigma_{Price} = \sigma_{YZ}(d-1) * O_Ref$$
$$\tilde{X}_{d}^{(s)} = \frac{X_{d}^{(s)} - O_{Ref,d}}{\sigma_{\text{Price}}}$$

where $X \in \{H, L, C, O\}$ and $s \in \{\text{Asia},\ \text{London}\}$.

In [6]:
sigma_price = pl.col("Sigma_Historical_Shifted") * pl.col("O_Ref")
_df = _df.with_columns([
    ((pl.col("H_Asia") - pl.col("O_Ref")) / sigma_price).alias("H_Asia_Normalized"),
    ((pl.col("L_Asia") - pl.col("O_Ref")) / sigma_price).alias("L_Asia_Normalized"),
    ((pl.col("C_Asia") - pl.col("O_Ref")) / sigma_price).alias("C_Asia_Normalized"),
    ((pl.col("O_London") - pl.col("O_Ref")) / sigma_price).alias("O_London_Normalized"),
    ((pl.col("H_London") - pl.col("O_Ref")) / sigma_price).alias("H_London_Normalized"),
    ((pl.col("L_London") - pl.col("O_Ref")) / sigma_price).alias("L_London_Normalized"),
    ((pl.col("C_London") - pl.col("O_Ref")) / sigma_price).alias("C_London_Normalized"),
])

***
#### **Economic Event Encoding**

For each trading day $d$, three event context features are constructed capturing the macro event weight on the preceding, current, and following calendar day:

$$\mathbf{n}_d = \left[\, e_{d-1},\ e_d,\ e_{d+1} \,\right] \in \{0, 1, 2, 3\}^3$$

where the daily event weight $e_t$ is defined as the maximum impact weight across all qualifying USD events on day $t$:

$$e_t = \max_{i \in \mathcal{E}_t} w_i, \qquad e_t = 0 \text{ if } \mathcal{E}_t = \emptyset$$

with impact weight tiers:

$$w_i = \begin{cases} 3 & \text{ultra-high: CPI, Core CPI, NFP, FOMC} \\ 2 & \text{high: Unemployment Claims, Average Hourly Earnings} \\ 1 & \text{medium: PPI, Core PPI, ADP, ISM Manufacturing,} \\ & \phantom{\text{medium: }} \text{ISM Services, JOLTs, Core Retail Sales} \\ 0 & \text{no qualifying event} \end{cases}$$

In [7]:
# load_dotenv()
# event_table = build_event_table(
#     aggregated_data.select(pl.col("Session").min()).item(),
#     aggregated_data.select(pl.col("Session").max()).item(),
#     os.getenv("ALFRED_API_KEY")
# )
# write_parquet(event_table, "event_table")

In [8]:
inspect = inspect_event_table(os.getenv("ALFRED_API_KEY"), date(2016, 1, 5), date(2016, 2, 10))
print(inspect.head(10))

  [ok] FOMC_SCRAPED         (US Federal Funds Rate): 0 meetings
shape: (10, 4)
┌────────────┬──────────┬───────────────┬────────────────────────────────┐
│ Session    ┆ e_weight ┆ series_id     ┆ name                           │
│ ---        ┆ ---      ┆ ---           ┆ ---                            │
│ date       ┆ i64      ┆ str           ┆ str                            │
╞════════════╪══════════╪═══════════════╪════════════════════════════════╡
│ 2016-01-06 ┆ 1        ┆ ISM_SVC       ┆ US ISM Services PMI            │
│ 2016-01-07 ┆ 2        ┆ ICSA          ┆ US Unemployment Claims         │
│ 2016-01-08 ┆ 2        ┆ CES0500000003 ┆ US Average Hourly Earnings m/m │
│ 2016-01-08 ┆ 1        ┆ MANEMP        ┆ US ISM Manufacturing PMI       │
│ 2016-01-08 ┆ 3        ┆ PAYEMS        ┆ US Non-Farm Employment Change  │
│ 2016-01-12 ┆ 1        ┆ JTSJOL        ┆ US JOLTS Job Openings          │
│ 2016-01-14 ┆ 2        ┆ ICSA          ┆ US Unemployment Claims         │
│ 2016-01-15 ┆ 1     

In [9]:
event_table = load_parquet(config.data.event_path)
_df = encode_news_context(_df, event_table)

In [10]:
event_table.head(5)

Session,e_weight
date,i8
2016-01-06,1
2016-01-07,2
2016-01-08,3
2016-01-12,1
2016-01-14,2


In [11]:
_df.filter(pl.col("Session") == date(2026, 2, 12)).select(["Session", "e_yesterday", "e_today", "e_tomorrow"])

Session,e_yesterday,e_today,e_tomorrow
date,i8,i8,i8
2026-02-12,3,2,3


***

The context table $\mathcal{C}$ is indexed by trading day $d \in \mathcal{D}$:

$$\mathcal{C}_d = \left[\,\tilde{H}_d^{A},\ \tilde{L}_d^{A},\ \tilde{C}_d^{A},\ \tilde{O}_d^{L},\ \tilde{H}_d^{L},\ \tilde{L}_d^{L},\ \tilde{C}_d^{L},\ \sigma_d,\ \sigma_{d-1},\ e_{d-1}, \ e_{d}, \ e_{d+1} \ \right] \in \mathbb{R}^{12}$$

where $d$ corresponds to the `Session` date key used for alignment with pivot sequences and targets.

In [12]:
# Columns that will go in context table

CONTEXT_WHITELIST = [
    "Sigma_Today", "Sigma_Historical_Shifted",
    "e_yesterday", "e_today", "e_tomorrow",
    "H_Asia_Normalized", "L_Asia_Normalized", "C_Asia_Normalized",
    "O_London_Normalized", "H_London_Normalized",
    "L_London_Normalized", "C_London_Normalized",
]

In [13]:
ctx_df = (
    _df.select(["Session", *CONTEXT_WHITELIST])
    .drop_nulls(["Session", *CONTEXT_WHITELIST])
    .sort("Session")
)

In [14]:
print(ctx_df.columns)

['Session', 'Sigma_Today', 'Sigma_Historical_Shifted', 'e_yesterday', 'e_today', 'e_tomorrow', 'H_Asia_Normalized', 'L_Asia_Normalized', 'C_Asia_Normalized', 'O_London_Normalized', 'H_London_Normalized', 'L_London_Normalized', 'C_London_Normalized']


In [15]:
ctx_df.filter(pl.col("Session") == date(2026, 2, 4))

Session,Sigma_Today,Sigma_Historical_Shifted,e_yesterday,e_today,e_tomorrow,H_Asia_Normalized,L_Asia_Normalized,C_Asia_Normalized,O_London_Normalized,H_London_Normalized,L_London_Normalized,C_London_Normalized
date,f64,f64,i8,i8,i8,f64,f64,f64,f64,f64,f64,f64
2026-02-04,0.00523,0.01182,0,1,2,0.422382,-0.084976,0.384893,0.382393,0.406553,-0.109969,0.022494


***
#### **Label Construction**
***
**Normalized Excursions**

This measures how far price moved up and down from NY Open, to account for regime changes it would be scaled by daily volatility.
$$z_{pos} = \frac{H_{NY} - O_{NY}}{\sigma_{YZ} \cdot O_{NY}}$$
$$z_{neg} = \frac{O_{NY} - L_{NY}}{\sigma_{YZ} \cdot O_{NY}}$$


In [16]:
eps = 1e-8 # just a very small number to prevent division by zero in normalization

z_pos = ((pl.col("H_New York") - pl.col("O_New York")).clip(lower_bound=0) / (pl.col("Sigma_Historical") * pl.col("O_New York") + eps))
z_neg = ((pl.col("O_New York") - pl.col("L_New York")).clip(lower_bound=0) / (pl.col("Sigma_Historical") * pl.col("O_New York") + eps))

label_df = (
    _df.select(["Session", "H_New York", "O_New York", "L_New York", "Sigma_Historical"]).with_columns([
        z_pos.alias("z_pos"),
        z_neg.alias("z_neg")
    ]).select(["Session", "z_pos", "z_neg"]).drop_nulls().sort("Session")
)

In [17]:
print(ctx_df.filter(pl.col("Session") == date(2026, 2, 4)))
print(label_df.filter(pl.col("Session") == date(2026, 2, 4)))

shape: (1, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Session   ┆ Sigma_Tod ┆ Sigma_His ┆ e_yesterd ┆ … ┆ O_London_ ┆ H_London_ ┆ L_London_ ┆ C_London │
│ ---       ┆ ay        ┆ torical_S ┆ ay        ┆   ┆ Normalize ┆ Normalize ┆ Normalize ┆ _Normali │
│ date      ┆ ---       ┆ hifted    ┆ ---       ┆   ┆ d         ┆ d         ┆ d         ┆ zed      │
│           ┆ f64       ┆ ---       ┆ i8        ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ f64       ┆           ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2026-02-0 ┆ 0.00523   ┆ 0.01182   ┆ 0         ┆ … ┆ 0.382393  ┆ 0.406553  ┆ -0.109969 ┆ 0.022494 │
│ 4         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
└───────────┴───────────┴───────────┴───────────┴───┴───────────┴───────────

***
#### **Transformer Dataset**
For each session (day) $d$, we construct a token matrix:
$$X_d \in \R^{T\times D}$$

$$T = 1 + max\_pivots$$
$$D = 1 + F_c + F_p + F_{cat}$$
where:
- 1 -> context token
- max_pivots -> number of maximum tokens
- $F_c$ -> Context columns count
- $F_p$ -> Pivots numerical columns count
- $F_{cat}$ -> Pivots categorical columns count
***
**Token Structure**

Each token $x_i \in \R^D$:
$$x_i = \begin{cases}
\left[1,\ c_d, \ 0, \ 0 \right] \qquad \text{if i = 0 (context)}\\
\left[0,\ 0, \ p_k, \ e_k \right] \qquad \text{if i = 0 (context)}\\
\end{cases}
$$
where:
- $c_d$ -> Context features
- $p_k$ -> Pivot numerical features
- $e_k$ -> Pivot categorical features

In [18]:
required = ["Session", *PIVOT_NUMERIC_WHITELIST, *PIVOT_CATEGORICAL_WHITELIST]
pivots_df = pivots_data.select(required)
pivots_df = pivots_df.with_columns(pl.int_range(pl.len()).over("Session").alias("global_pos"))

In [19]:
valid_sessions = (
    ctx_df.select("Session")
    .join(label_df.select("Session"), on="Session", how="inner")
    .join(pivots_df.select("Session").unique(), on="Session", how="inner")
    .sort("Session")
)
# to skip burn in period
valid_sessions = valid_sessions.filter(pl.col("Session") >= date(2016, 4, 5))["Session"].to_list()

In [20]:
print(valid_sessions[:5])

[datetime.date(2016, 4, 5), datetime.date(2016, 4, 6), datetime.date(2016, 4, 7), datetime.date(2016, 4, 8), datetime.date(2016, 4, 11)]


In [21]:
pivots_df.filter(pl.col("Session") == date(2026, 2, 4)).tail(5)

Session,Pi_k,Delta_FE_Pos,Delta_AE_Pos,Delta_FE_Neg,Delta_AE_Neg,State_AE_Neg,State_AE_Pos,State_FE_Neg,State_FE_Pos,delta_Pi_k,delta_b_k,Speed_k,Dir_k,Turn_k,s_k,Intraday_Session,global_pos
date,f64,f64,f64,f64,f64,i8,i8,i8,i8,f64,i16,f64,i32,i8,i32,str,i64
2026-02-04,0.070814,-1.26085,-0.405625,1.402477,0.547252,0,0,0,0,-0.130797,2,-0.065398,-1,0,-1,"""Asia""",6
2026-02-04,0.422382,-0.909282,-0.054057,1.754046,0.898821,0,1,0,0,0.351569,4,0.087892,1,1,1,"""Asia""",7
2026-02-04,-0.109969,-1.441633,-0.586408,1.221694,0.366469,0,0,0,0,-0.532351,4,-0.133088,-1,1,-1,"""London""",8
2026-02-04,0.262427,-1.069237,-0.214012,1.59409,0.738866,0,0,0,0,0.372396,2,0.186198,1,1,1,"""London""",9
2026-02-04,0.343238,-0.988426,-0.133201,1.674901,0.819676,0,0,0,0,0.080811,3,0.026937,1,0,1,"""London""",10


In [22]:
intraday_session_map = {"Asia": 1, "London": 2}
s_k_map = {-1: 1, 1: 2}

Fc = len(CONTEXT_WHITELIST)
Fp = len(PIVOT_NUMERIC_WHITELIST)
Fcat = len(PIVOT_CATEGORICAL_WHITELIST)

max_pivots = 27

# 1 here is accounting for context flag
token_dim = 1 + Fc + Fp + Fcat
T = 1 + max_pivots


***
**Session Matrix**

For each session $d$:
$$X_{d} = 
\begin{bmatrix}
1 & c_1 & \cdot\cdot\cdot & c_{F_{c}} & 0 & \cdot\cdot\cdot & 0 & 0 & \cdot\cdot\cdot & 0\\
0 & 0 & \cdot\cdot\cdot & 0 & p_{1,1} & \cdot\cdot\cdot & p_{1, j} & e_{1, 1} & \cdot\cdot\cdot & e_{1, k}\\
0 & 0 & \cdot\cdot\cdot & 0 & p_{1,1} & \cdot\cdot\cdot & p_{2, N} & e_{2, 1} & \cdot\cdot\cdot & e_{2, k}\\
\vdots & \vdots & & \vdots & \vdots & & \vdots & \vdots & & \vdots\\
0 & 0 & \cdot\cdot\cdot & 0 & p_{i,1} & \cdot\cdot\cdot & p_{i, j} & e_{i, 1} & \cdot\cdot\cdot & e_{i, k}\\
\vdots & \vdots & & \vdots & \vdots & & \vdots & \vdots & & \vdots\\
\end{bmatrix}
$$

where:
- $p_{i, j}$ = Pivots numerical values
- $e_{i, k}$ = Pivots categorical values

<br>

**Attention Mask**:
$$m_d \in \{ 0,1\}^T$$
$$m_i = \begin{cases}
1 \qquad token\\
0 \qquad padding
\end{cases}
$$
This mask is to distinguish which are real tokens and which are just padding, we use padding since our number of pivots are not fixed.

<br>

**Position**:

For each session:
$$p_d \in \N^T$$
$$p_i = \begin{cases}
0 \qquad \text{Context Token}\\
k \qquad \text{Pivot Index}\\
0 \qquad \text{Padding}
\end{cases}
$$

These are positions for tokens inside a single session window

<br>

**Labels**:

$$y_d = [z_{pos}, \ z_{neg}]$$

where:
- $z_{pos}$ = Normalized positive excursion from $O_{NY}$
- $z_{neg}$ = Normalized negative excursion from $O_{NY}$

In [23]:
# diagnostics variables
max_seen_pivots = 0
truncated_days = 0

In [24]:
x_tokens, mask, position = [], [], []
z_pos_labels, z_neg_labels = [], []
sessions = []

In [25]:
for sess in valid_sessions:
    c_row = ctx_df.filter(pl.col("Session") == sess)
    p_day = pivots_df.filter(pl.col("Session") == sess)
    l_row = label_df.filter(pl.col("Session") == sess)

    if c_row.height != 1 or l_row.height != 1:
        continue

    max_seen_pivots = max(max_seen_pivots, p_day.height)
    if p_day.height > max_pivots:
        truncated_days += 1
        # if max pivots then older pivots are truncated
        p_day = p_day.tail(max_pivots)

    x = np.zeros((T, token_dim), dtype=np.float32)
    m = np.zeros((T,), dtype=np.int64)
    p = np.zeros((T,), dtype=np.int64)

    ctx_vals = c_row.select(CONTEXT_WHITELIST).to_numpy().astype(np.float32)[0]
    # flag for context row
    x[0, 0] = 1.0

    x[0, 1:1+Fc] = ctx_vals
    m[0] = 1
    p[0] = 0

    i = 1
    for row in p_day.iter_rows(named=True):
        if i >= T:
            break

        p_num = np.array([float(row[col]) for col in PIVOT_NUMERIC_WHITELIST], dtype=np.float32)
        s_k_encoded = s_k_map.get(int(row["s_k"]), 0)
        intraday_session_encoded = intraday_session_map.get(str(row["Intraday_Session"]), 0)

        start = 1 + Fc
        x[i, start:start+Fp] = p_num
        x[i, start+Fp] = float(s_k_encoded)
        x[i, start+Fp+1] = float(intraday_session_encoded)

        m[i] = 1
        p[i] = int(row["global_pos"]) + 1
        i += 1
    x_tokens.append(x)
    mask.append(m)
    position.append(p)
    z_pos_labels.append(float(l_row["z_pos"][0]))
    z_neg_labels.append(float(l_row["z_neg"][0]))
    sessions.append(sess)

In [26]:
dataset = {
    "X_Tokens": np.stack(x_tokens, axis=0),
    "Attention_Mask": np.stack(mask, axis=0),
    "Token_Position": np.stack(position, axis=0),
    "Z_Pos_Labels": np.array(z_pos_labels, dtype=np.float32),
    "Z_Neg_Labels": np.array(z_neg_labels, dtype=np.float32),
    "Sessions": sessions,
    "Features_Metadata": {
        "Max_Pivots": max_pivots,
        "Height": T,
        "Token_Dim": token_dim,
        "Truncated_Days": truncated_days,
        "Max_Seen_Pivots": max_seen_pivots,
    }
}

***